# 02 — Weather & Database
Building a DuckDB database and pulling weather data for every Mets home game from the Open-Meteo API (free, no API key needed).

In [ ]:
import pandas as pd
import requests

df = pd.read_csv("../data/mets_games_raw.csv")
print(f"Loaded {len(df)} games")
df_2026 = pd.read_csv("../data/mets_2026_prediction.csv")
print(f"Loaded {len(df_2026)} 2026 games")
df.head()

## Pulling Weather for 2022-2025 Game Dates
Citi Field coordinates are latitude 40.7571, longitude -73.8458.

Note: New York weather varies significantly — cold April/May games, summer heat and humidity, and rain are all meaningful signals unlike San Diego. We pull hourly temperature and precipitation around game time (4pm-8pm local time).

In [ ]:
def get_weather(date, lat=40.7571, lon=-73.8458):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date,
        "end_date": date,
        "hourly": "temperature_2m,precipitation",
        "temperature_unit": "fahrenheit",
        "timezone": "America/New_York"
    }
    resp = requests.get(url, params=params).json()
    hourly = resp.get("hourly", {})

    # Get conditions around game time (4pm-8pm = hours 16-20)
    temps = [t for t in hourly.get("temperature_2m", [])[16:21] if t is not None]
    precip = [p for p in hourly.get("precipitation", [])[16:21] if p is not None]

    return {
        "date": date,
        "avg_temp_f": round(sum(temps) / len(temps), 1) if temps else None,
        "avg_precip_mm": round(sum(precip) / len(precip), 1) if precip else None
    }

dates = df["date"].unique()
print(f"Pulling weather for {len(dates)} unique game dates...")

weather_data = [get_weather(d) for d in dates]
df_weather = pd.DataFrame(weather_data)
print("Done!")
df_weather.head()

In [ ]:
df_weather.to_csv("../data/mets_weather.csv", index=False)
print(f"Saved weather for {len(df_weather)} dates")
print(df_weather.describe())

In [ ]:
# Pull weather for 2026 games already played
dates_2026 = df_2026[df_2026['status'] == 'Final']['date'].unique()
print(f"Pulling weather for {len(dates_2026)} 2026 game dates...")

weather_2026 = [get_weather(d) for d in dates_2026]
df_weather_2026 = pd.DataFrame(weather_2026)
df_weather_2026.to_csv("../data/mets_weather_2026.csv", index=False)
print("Done!")
df_weather_2026.describe()

## Build DuckDB Database
Load games and weather data into a DuckDB database. DuckDB uses standard SQL syntax and is optimized for analytical queries — similar to how BigQuery works in production.

In [ ]:
import duckdb

con = duckdb.connect("../data/mets.duckdb")

con.execute("CREATE OR REPLACE TABLE games AS SELECT * FROM read_csv_auto('../data/mets_games_raw.csv')")
con.execute("CREATE OR REPLACE TABLE weather AS SELECT * FROM read_csv_auto('../data/mets_weather.csv')")

print("Tables created:")
print(con.execute("SHOW TABLES").fetchdf())
print("Games rows:", con.execute("SELECT COUNT(*) FROM games").fetchone()[0])
print("Weather rows:", con.execute("SELECT COUNT(*) FROM weather").fetchone()[0])

In [ ]:
df_combined = con.execute("""
    SELECT
        g.date,
        g.season,
        g.opponent,
        g.home_score,
        g.away_score,
        g.attendance,
        g.status,
        g.game_pk,
        w.avg_temp_f,
        w.avg_precip_mm
    FROM games g
    LEFT JOIN weather w ON g.date = w.date
    WHERE g.status = 'Final'
    ORDER BY g.date
""").fetchdf()

print(f"Combined dataset: {len(df_combined)} games")
print(df_combined.head())

In [ ]:
df_combined.to_csv("../data/mets_master.csv", index=False)
print("Saved mets_master.csv")